# 04 · NumPy 数值运算

用 NumPy 数组（array）取代 Python 串列（list），学会以矢量化（vectorization）思维做高效率数值运算。

## 学习目标
- 创建并理解 NumPy 数组的形状（shape）、数据类型（dtype），能变形（reshape）与拼接（concatenate）。
- 掌握广播（broadcasting）规则，用形状不同的数组做整批运算。
- 运用布尔遮罩（boolean mask）与进阶索引（advanced indexing）做筛选与条件赋值。
- 使用统计与运算函数（mean、std、median、percentile、where、argsort 等）萃取数据特征。
- 认识线性代数（linear algebra）与随机乱数（random）模块，并能设置种子（seeding）重现结果。
- 体会矢量化相对于循环（loop）的性能差异，培养避免明写循环的习惯。

In [ ]:
# 中文字体设置（本书会画图才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 数组创建与形状

NumPy 数组（array）是一块「同类型、有形状」的连续数据，是后续一切数值运算的基础。
相较于 Python 串列，数组记住自己的形状（shape）与数据类型（dtype），让整批运算又快又省内存。

先创建「形状与 dtype 决定运算行为」的直觉，并区分两件常被搞混的事：
- 变形（reshape）：总元素数不变，只重新安排成不同维度。
- 拼接（concatenate）/ 垂直堆栈（vstack）：把多块数组接成更大的一块。

形状：`arr.reshape(列数, 栏数)`，其中 `列数 × 栏数` 必须等于原本的元素总数。

In [ ]:
# 概念：数组创建 array creation、形状 shape、变形 reshape、拼接 concatenate
import numpy as np

nums = np.arange(12)              # 造一串 0..11 的整数当示范用
print('原始一维数组：', nums)
print('shape =', nums.shape, ' dtype =', nums.dtype)

grid = nums.reshape(3, 4)         # 变形成 3 列 4 栏，元素总数 12 不变
print('reshape 后：\n', grid)

# 造两块小数组，准备示范堆栈与拼接
top = np.array([[1, 1, 1, 1]])    # 形状 (1, 4)
bottom = np.array([[9, 9, 9, 9]])

# 注意：vstack 沿「列」方向往下叠，栏数要一致
stacked = np.vstack([top, grid, bottom])
print('vstack 叠出的矩阵：\n', stacked)

# 技巧：concatenate 用 axis 指定方向，axis=0 等同 vstack
joined = np.concatenate([top, bottom], axis=0)
print('concatenate(axis=0)：\n', joined)

## 广播

广播（broadcasting）让形状不同的数组免写循环即可逐元素运算，是 NumPy 高效率的核心机制。
运算时 NumPy 会把较小的数组「自动延展」到对齐较大的形状（形状对齐 shape alignment）。

规则直觉：从最后一个维度往前比，对应维度若相等、或其中一个为 1，就能延展；否则失败报错。
纯量（scalar）与数组运算是最常见的广播，例如「整批加上同一个数」。

In [ ]:
# 概念：广播 broadcasting、形状对齐 shape alignment、纯量与数组运算
import numpy as np

# 造一组假数据：4 列观测、3 栏特征
data = np.array([[10.0, 20.0, 30.0],
                 [12.0, 18.0, 33.0],
                 [11.0, 25.0, 27.0],
                 [13.0, 22.0, 30.0]])

col_mean = data.mean(axis=0)      # 沿列方向算每一栏的平均，得到形状 (3,)
print('各栏平均：', col_mean)

# 注意：(4, 3) 与 (3,) 从最后维度对齐，3==3 可广播；列矢量自动套到每一列
centered = data - col_mean        # 每栏都减去自己的平均（去中心化）
print('去中心化后：\n', centered)

# 技巧：纯量也是广播，整批加 100 不需循环
print('整批加 100：\n', data + 100)

## 布尔遮罩与进阶索引

布尔遮罩（boolean mask）是一个由 True / False 组成、形状与数据相同的数组。
用 `arr[mask]` 可一次取出所有 True 位置的元素，取代逐笔判断的循环，是数据清理与筛选的关键手法。

同样可做条件赋值（conditional assignment）：`arr[mask] = 新值`，把符合条件的元素整批改掉。

形状：`mask = arr > 门槛` 产生遮罩；`arr[mask]` 取值；`arr[mask] = 0` 改值。

In [ ]:
# 概念：布尔遮罩 boolean mask、条件筛选 arr[mask]、条件赋值 conditional assignment
import numpy as np

# 造一组假分数（含一个用来示范清理的负值）
scores = np.array([72, 58, 91, -5, 66, 45, 88])

pass_mask = scores >= 60          # 及格门槛 60，产生布尔遮罩
print('遮罩：', pass_mask)
print('及格者：', scores[pass_mask])  # 一次取出所有 True 的元素

# 注意：条件赋值会就地修改原数组，把不合理的负值统一改为 0
scores[scores < 0] = 0
print('清理负值后：', scores)

# 技巧：多条件要用 & | 并各自加括号（不能用 and / or）
band_mask = (scores >= 60) & (scores < 90)
print('60~89 分：', scores[band_mask])

## 统计摘要

统计摘要把一堆数字浓缩成少数几个描述分布的特征：平均（mean）、标准差（std）、中位数（median）、总和（sum）、百分位数（percentile）。

关键在于轴（axis）：对二维数组，`axis=0` 沿列方向逐栏计算，`axis=1` 沿栏方向逐列计算，不指定则对全体计算。
平均易受极端值拉动，中位数较稳健，两者并看能更了解数据分布。

In [ ]:
# 概念：平均 mean、标准差 std、中位数 median、百分位数 percentile、轴 axis
import numpy as np

# 造一个 3 列 4 栏的假数值矩阵
matrix = np.array([[ 5.0,  8.0,  6.0,  9.0],
                   [ 7.0, 50.0,  6.0,  8.0],
                   [ 6.0,  7.0,  5.0, 10.0]])

print('整体平均：', matrix.mean())
print('整体标准差：', round(matrix.std(), 3))

# 注意：axis=0 是「跨列、逐栏」，结果长度等于栏数 4
print('逐栏平均：', matrix.mean(axis=0))
print('逐栏中位数：', np.median(matrix, axis=0))

# 技巧：percentile 一次给多个分位点，回传对应的数组
print('第 25/50/75 百分位数：', np.percentile(matrix, [25, 50, 75]))

## 运算与转换函数

这组函数让常见的特征处理都能矢量化完成：
- 裁切（clip）：把值限制在上下界之内。
- 条件选择（where）：依条件逐元素在两个选项间挑一个。
- 排序索引（argsort）：回传「排序后的索引」，常用来找名次。
- 最大值索引（argmax）：回传最大值所在的位置。
- 次数统计（bincount）：统计非负整数标签各自出现几次。

In [ ]:
# 概念：裁切 clip、条件选择 where、排序索引 argsort、最大值索引 argmax、次数统计 bincount
import numpy as np

values = np.array([3, -2, 18, 7, 25, 5, 9])

clipped = np.clip(values, 0, 20)  # 小于 0 变 0、大于 20 变 20
print('裁切到 [0, 20]：', clipped)

# 条件选择：达标标 1、否则标 0
flags = np.where(values >= 10, 1, 0)
print('是否达 10：', flags)

# 注意：argsort 预设由小到大，取最大前三名要反转索引
order_desc = np.argsort(values)[::-1]
print('最大前三名的索引：', order_desc[:3])
print('最大值所在位置 argmax：', np.argmax(values))

# 技巧：bincount 统计类别标签次数（标签须为非负整数）
labels = np.array([0, 1, 2, 1, 0, 1, 2])
print('各类别次数：', np.bincount(labels))

## 线性代数与随机乱数

NumPy 内置线性代数（linear algebra）与随机乱数（random）模块。
- 矢量范数（linalg.norm）：衡量矢量长度或残差大小。
- 最小平方解（linalg.lstsq）：在过定方程中求最佳拟合直线。
- 乱数产生器：建议用新式 `default_rng`（旧式为 `RandomState`），并以种子（seeding）固定结果。

设置种子能让每次运行得到相同的「随机」数据，是实验可重现（reproducibility）的关键。

In [ ]:
# 概念：乱数产生器 default_rng、种子 seeding、最小平方解 linalg.lstsq、矢量范数 linalg.norm
import numpy as np

rng = np.random.default_rng(42)   # 固定种子 42，确保结果可重现

x = np.linspace(0, 10, 30)        # 0 到 10 取 30 个等距点
noise = rng.normal(0, 1.0, size=x.shape)  # 平均 0、标准差 1 的高斯杂讯
y = 2.0 * x + 1.0 + noise         # 真实关系 y=2x+1 再加杂讯

# 注意：lstsq 需要设计矩阵，第二栏补 1 代表截距项
A = np.column_stack([x, np.ones_like(x)])
(slope, intercept), *_ = np.linalg.lstsq(A, y, rcond=None)
print('拟合斜率：', round(slope, 3), ' 截距：', round(intercept, 3))

residual = y - (slope * x + intercept)
print('残差范数：', round(np.linalg.norm(residual), 3))

plt.scatter(x, y, label='带杂讯的点')
plt.plot(x, slope * x + intercept, color='red', label='lstsq 拟合直线')
plt.xlabel('x')
plt.ylabel('y')
plt.title('最小平方拟合')
plt.legend()
plt.show()

## 矢量化与性能

矢量化（vectorization）是把整批运算交给 NumPy 底层的 C 实作一次完成，免去 Python 层的逐笔循环（loop）。
数据量大时，矢量化通常比明写循环快上数十到数百倍，这是 NumPy 的主要价值。

工程习惯：动手写循环前，先想「这能不能矢量化」。下面用计时（timing）比较同一计算的两种写法。

In [ ]:
# 概念：矢量化 vectorization、循环 loop、计时 timing、性能差异 performance
import numpy as np
import time

big = np.arange(1_000_000, dtype=np.float64)  # 造一个百万元素的大数组

# 写法一：Python 循环逐笔累加平方
t0 = time.perf_counter()
loop_sum = 0.0
for v in big:
    loop_sum += v * v
loop_time = time.perf_counter() - t0

# 写法二：矢量化，一行算完所有平方再加总
t0 = time.perf_counter()
vec_sum = np.sum(big * big)
vec_time = time.perf_counter() - t0

print('循环结果：', loop_sum, ' 耗时：', round(loop_time, 4), '秒')
print('矢量化结果：', vec_sum, ' 耗时：', round(vec_time, 4), '秒')
# 技巧：结果相同但矢量化快很多，倍率视机器而定
print('矢量化约快', round(loop_time / max(vec_time, 1e-9), 1), '倍')

## 练习

以下三题由浅入深，皆为复合型但简短可快速完成。数据一律在题目内用 numpy 自造，不引用任何外部文件。

In [ ]:
# TODO 1 ·（简单）计算建筑楼地板面积与容积率（集成：数组创建 array creation + 广播 broadcasting）
#   情境：手上有一小批建筑的占地面积 footprint area 与楼层数 floors，
#         想快速算出每栋的楼地板面积 GFA 与容积率 FAR。
#   要求：
#     1. 用 numpy 自造两个形状相同的一维数组：footprint（占地面积，平方公尺，5 栋）与 floors（楼层数）。
#     2. 用逐元素相乘（广播）算出楼地板面积 GFA = footprint * floors。
#     3. 自造一维数组 lot（各栋基地面积），用 GFA / lot 算出容积率 FAR。
#   验收：应看到一个与输入等长的 FAR 数组，每个值都是合理正数（例如 1.0 到 5.0 之间）。
# TODO: 在这里完成

In [ ]:
# TODO 2 ·（中间）街廓网格的建筑高度聚合
#   （集成：数组创建 array creation + 广播 broadcasting + 布尔遮罩 boolean mask + 统计摘要 statistical summary）
#   情境：把一个街廓 block 切成数个网格 cell，需统整每格内置筑楼高 height 的分布，并标记过高的建筑。
#   要求：
#     1. 用 numpy 自造一个二维数组 heights（形状 4 列 x 6 栏，代表 4 格各 6 栋建筑的楼高，公尺）。
#     2. 沿轴 axis 算出每一格（每列）的平均 mean、标准差 std 与第 90 百分位数 percentile。
#     3. 用广播把每栋楼高减去「该格平均」，得到每栋相对于所在网格的高度差。
#     4. 用布尔遮罩找出全体中超过「整体平均 + 一个标准差」的过高建筑，并回报其数量。
#   验收：应看到每格的三项统计值，以及一个过高建筑的布尔遮罩与其总数。
# TODO: 在这里完成

In [ ]:
# TODO 3 ·（稍难）政策高度乘数下的容积达标筛选
#   （集成：数组创建 array creation + 广播 broadcasting + 条件选择 where + 统计摘要 statistical summary + 排序索引 argsort）
#   情境：某政策对不同用途分类 land-use 给予不同高度乘数 height multiplier，
#         需评估调整后哪些建筑达到目标容积率 FAR，并排出潜力名单。
#   要求：
#     1. 用 numpy 自造：floors（楼层数）、footprint（占地面积）、lot（基地面积），
#        以及整数数组 usecode（用途标签，0=住宅、1=商业、2=工业）。
#     2. 建长度 3 的乘数数组 multiplier，用进阶索引 multiplier[usecode] 把每栋对应到自己的乘数，
#        再用 where 对特定用途（如商业）再加成，得到调整后的有效楼层数。
#     3. 用广播算出调整后 GFA 与 FAR，并用布尔遮罩筛出 FAR 达门槛（如 >= 3.0）的建筑。
#     4. 对达标建筑依 FAR 由高到低用 argsort 排序，输出前 3 名的索引作为开发潜力名单。
#   验收：应看到达标建筑的布尔遮罩、达标数量，以及依 FAR 排序后的前 3 名建筑索引。
# TODO: 在这里完成

## 小结

- NumPy 数组以形状（shape）与数据类型（dtype）为核心，reshape 改排列、concatenate / vstack 接数据。
- 广播（broadcasting）让不同形状的数组免循环逐元素运算，是高效率的基础机制。
- 布尔遮罩（boolean mask）与进阶索引（advanced indexing）用条件直接取值与改值，取代逐笔判断。
- 统计函数搭配轴（axis）能逐栏 / 逐列萃取平均、标准差、中位数与百分位数等特征。
- clip、where、argsort、argmax、bincount 是常用的矢量化特征处理工具。
- 线性代数与乱数模块搭配种子（seeding）可做可重现的拟合与实验。
- 动手写循环前先想能否矢量化，是写出高效数值程序的工程习惯。